In [12]:
import os
import json
import time
import requests
from dotenv import load_dotenv

# ✅ Load API key
load_dotenv()
API_KEY = os.getenv("GOOGLE_MAP_API_KEY")
if not API_KEY:
    raise ValueError("❌ GOOGLE_MAP_API_KEY not found in environment.")

# --- CONFIG ---
SOURCE_URL = "https://raw.githubusercontent.com/ThathsaraniPathirana/LLM-project/refs/heads/main/Food_Establishment/all_food_sweden_flat.json"
OUTPUT_FILE = "ratings_food.json"
BATCH_SIZE = 50
SLEEP_SEC = 1

# --- Download source data ---
print("📂 Downloading source JSON...")
resp = requests.get(SOURCE_URL)
resp.raise_for_status()
data = resp.json()
print(f"✅ Loaded {len(data)} food establishments.")

# --- Resume progress if available ---
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        ratings = json.load(f)
    processed_names = {r["name"] for r in ratings}
    print(f"🔁 Resuming — already processed {len(processed_names)} records.")
else:
    ratings, processed_names = [], set()

# --- Google Places API setup ---
url = "https://places.googleapis.com/v1/places:searchText"
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.rating,places.userRatingCount,places.googleMapsUri"
}

# --- Helper: extract region name from URL ---
def extract_region_name(region_url):
    if not region_url:
        return None
    return region_url.split("/")[-1].replace("-", " ").title()

# --- Helper: get safe string from mixed type ---
def safe_value(val):
    """Handles dicts like {'@value': 'Furuvik'} and plain strings."""
    if isinstance(val, dict):
        return val.get("@value")
    elif isinstance(val, str):
        return val.strip()
    else:
        return None

# --- Main loop ---
count = 0
for record in data:
    name = record.get("name")
    alt = record.get("alternate_name")
    street = safe_value(record.get("street"))
    city = safe_value(record.get("city"))
    region = record.get("region")

    if not name or name in processed_names:
        continue

    # Build a clean search query
    region_name = extract_region_name(region)
    parts = [name, alt, street, city, region_name, "Sweden"]
    query = ", ".join(str(p) for p in parts if p)
    payload = {"textQuery": query}

    try:
        res = requests.post(url, headers=headers, json=payload, timeout=15)
        result = res.json()

        if "error" in result:
            print(f"⚠️ API Error for {name}: {result['error'].get('message')}")
            continue

        places = result.get("places", [])
        if not places:
            print(f"❌ No result found for: {name}")
            continue

        place = places[0]
        entry = {
            "name": name,
            "alternate_name": alt,  # keep as in input (can be null)
            "formattedAddress": place.get("formattedAddress"),
            "googleMapsUri": place.get("googleMapsUri"),
            "rating": place.get("rating"),
            "userRatingCount": place.get("userRatingCount")
        }

        ratings.append(entry)
        processed_names.add(name)
        count += 1

        print(f"✅ [{count}] {name} → ⭐ {entry['rating']} ({entry['userRatingCount']})")

        # Save every batch
        if count % BATCH_SIZE == 0:
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(ratings, f, ensure_ascii=False, indent=2)
            print(f"💾 Progress saved — {count} done.")
            time.sleep(2)

        time.sleep(SLEEP_SEC)

    except Exception as e:
        print(f"⚠️ Error for {name}: {e}")
        time.sleep(3)
        continue

# --- Final save ---
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(ratings, f, ensure_ascii=False, indent=2)

print(f"\n🎯 Completed! Saved {len(ratings)} entries to {OUTPUT_FILE}")


📂 Downloading source JSON...
✅ Loaded 1286 food establishments.
✅ [1] Burger King → ⭐ 3.5 (1120)
✅ [2] Pizzeria Empoli → ⭐ 4 (197)
✅ [3] Subway → ⭐ 4.8 (90)
✅ [4] Pizza House → ⭐ 4.4 (280)
✅ [5] McDonalds → ⭐ 2.9 (2664)
✅ [6] Flying Restaurang & Pub → ⭐ 4.1 (28)
✅ [7] Restaurang Hedåsen → ⭐ 4.3 (6)
✅ [8] Brasserie Draken → ⭐ 4.1 (164)
✅ [9] Stilleben Kök → ⭐ 4.8 (111)
✅ [10] Restaurang Sjöbacken → ⭐ 4.1 (388)
✅ [11] Säbyås gård → ⭐ 4.7 (7)
✅ [12] Ockelbo Kyckling - gårdsbutik → ⭐ 4.9 (32)
✅ [13] Nyholms Lantgård → ⭐ None (None)
✅ [14] Jensas Grill & Restaurang → ⭐ 4.1 (228)
✅ [15] Skråmträsk Kvarn → ⭐ 4.3 (305)
✅ [16] Restaurang Mandel → ⭐ 4 (103)
✅ [17] Wärdshuset Gamla Linköping → ⭐ 3.8 (265)
✅ [18] The Social → ⭐ 3.7 (119)
✅ [19] Verovin Vinbar → ⭐ 4.8 (290)
✅ [20] Böna Café → ⭐ 4.3 (1253)
✅ [21] Matildas → ⭐ 4.7 (420)
✅ [22] Furuvik Havskrog → ⭐ 4 (440)
✅ [23] Church Street Saloon → ⭐ 4.4 (1533)
✅ [24] Dellenbadens Kanalcafé "Hälsinglands största Glassbar" → ⭐ 4.4 (183)
✅ [25] Pink